# 1. Intro
This notebook transforms the raw OULAD datasets into a student-level modeling dataset.

The objectives are:

• Create informative features from assessment, registration, and engagement data.
• Merge information from multiple datasets.
• Remove target leakage variables.
• Produce the final modeling dataset for preprocessing and machine learning.

Output:
final_modeling_dataset.csv

# 2. Importing Libraries

In [289]:
import pandas as pd
import numpy as np

# 3. Load Raw OULAD Datasets

In [290]:
import kagglehub
import pandas as pd
path = kagglehub.dataset_download(
    "anlgrbz/student-demographics-online-education-dataoulad"
)

print("Path to dataset:", path)
df_students = pd.read_csv(path + "\\studentInfo.csv")
df_studentAssessment = pd.read_csv(path + "\\studentAssessment.csv")
df_assessments = pd.read_csv(path + "\\assessments.csv")
df_studentRegistration = pd.read_csv(path + "\\studentRegistration.csv")
df_studentVle = pd.read_csv(path + "\\studentVle.csv")
df_vle = pd.read_csv(path + "\\vle.csv")
df_courses = pd.read_csv(path + "\\courses.csv")

print("\nAll datasets loaded successfully.")

print("Dataset Shapes")

print(f"studentInfo           : {df_students.shape}")
print(f"studentAssessment     : {df_studentAssessment.shape}")
print(f"assessments           : {df_assessments.shape}")
print(f"studentRegistration   : {df_studentRegistration.shape}")
print(f"studentVle            : {df_studentVle.shape}")
print(f"vle                   : {df_vle.shape}")
print(f"courses               : {df_courses.shape}")

Path to dataset: C:\Users\jaypr\.cache\kagglehub\datasets\anlgrbz\student-demographics-online-education-dataoulad\versions\1

All datasets loaded successfully.
Dataset Shapes
studentInfo           : (32593, 12)
studentAssessment     : (173912, 5)
assessments           : (206, 6)
studentRegistration   : (32593, 5)
studentVle            : (10655280, 6)
vle                   : (6364, 6)
courses               : (22, 3)


# 4. Create Binary Target Variable

The original `final_result` column contains four possible student outcomes:

- Pass
- Fail
- Distinction
- Withdrawn

Since the objective of this project is to predict student dropout, the target variable is converted into a binary classification problem:

- **Withdrawn = 1** (Student Dropped Out)
- **Pass, Fail, Distinction = 0** (Student Completed the Course)

The original `final_result` column is retained for reference, while a new binary target column named `dropout_target` is created.

In [291]:
# Original Student Outcomes

print("Original Final Result Distribution:")
print(df_students["final_result"].value_counts())

print("\nPercentage Distribution:")
print(
    (df_students["final_result"].value_counts(normalize=True) * 100)
    .round(2)
    .astype(str) + "%"
)

Original Final Result Distribution:
final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64

Percentage Distribution:
final_result
Pass           37.93%
Withdrawn      31.16%
Fail           21.64%
Distinction     9.28%
Name: proportion, dtype: str


In [292]:
# Create Binary Dropout Target

df_students["dropout_target"] = (
    df_students["final_result"] == "Withdrawn"
).astype(int)

print("Binary target variable created successfully.")

Binary target variable created successfully.


In [293]:
# Verify Binary Target

print("Dropout Target Distribution:")

print(df_students["dropout_target"].value_counts())

print("\nPercentage Distribution:")

print(
    (df_students["dropout_target"].value_counts(normalize=True) * 100)
    .round(2)
    .astype(str) + "%"
)

Dropout Target Distribution:
dropout_target
0    22437
1    10156
Name: count, dtype: int64

Percentage Distribution:
dropout_target
0    68.84%
1    31.16%
Name: proportion, dtype: str


In [294]:
# Verify Mapping

mapping_check = pd.crosstab(
    df_students["final_result"],
    df_students["dropout_target"],
    margins=True
)

display(mapping_check)

dropout_target,0,1,All
final_result,,,
Distinction,3024,0,3024
Fail,7052,0,7052
Pass,12361,0,12361
Withdrawn,0,10156,10156
All,22437,10156,32593


In [295]:
# Preview Updated Dataset

display(
    df_students[
        [
            "id_student",
            "code_module",
            "code_presentation",
            "final_result",
            "dropout_target",
        ]
    ].head()
)

,id_student,code_module,code_presentation,final_result,dropout_target
0,11391,AAA,2013J,Pass,0
1,28400,AAA,2013J,Pass,0
2,30268,AAA,2013J,Withdrawn,1
3,31604,AAA,2013J,Pass,0
4,32885,AAA,2013J,Pass,0


# 5. Calculate 25% Course Checkpoint

The objective of this project is to predict student dropout using only the information available during the first **25% of the course duration**.

To achieve this, a checkpoint is calculated for each course presentation based on its total duration. This checkpoint will later be used to filter student assessment submissions and VLE interactions so that only early-course behaviour is considered during feature engineering.

Checkpoint Formula:

Checkpoint Day = 25% × Module Presentation Length

For example, if a course lasts 240 days, the prediction checkpoint is Day 60.

Create Checkpoint Dataset

In [296]:
# Create 25% Course Checkpoint

course_checkpoint = df_courses.copy()

course_checkpoint["checkpoint_day"] = (
    course_checkpoint["module_presentation_length"] * 0.25
).round().astype(int)

print("25% checkpoint calculated successfully.")
display(course_checkpoint)

25% checkpoint calculated successfully.


,code_module,code_presentation,module_presentation_length,checkpoint_day
0,AAA,2013J,268,67
1,AAA,2014J,269,67
2,BBB,2013J,268,67
3,BBB,2014J,262,66
4,BBB,2013B,240,60
5,BBB,2014B,234,58
6,CCC,2014J,269,67
7,CCC,2014B,241,60
8,DDD,2013J,261,65
9,DDD,2014J,262,66


Checkpoint Summary

In [297]:
# Checkpoint Summary

print("Course Checkpoint Summary")

print(f"Total Course Presentations : {len(course_checkpoint)}")

print(f"\nMinimum Course Duration : {course_checkpoint['module_presentation_length'].min()} days")

print(f"Maximum Course Duration : {course_checkpoint['module_presentation_length'].max()} days")

print(f"\nMinimum Checkpoint Day : {course_checkpoint['checkpoint_day'].min()}")

print(f"Maximum Checkpoint Day : {course_checkpoint['checkpoint_day'].max()}")

Course Checkpoint Summary
Total Course Presentations : 22

Minimum Course Duration : 234 days
Maximum Course Duration : 269 days

Minimum Checkpoint Day : 58
Maximum Checkpoint Day : 67


Validate Checkpoint Dataset

In [298]:
# Validate Checkpoint Dataset

print("=" * 60)
print("Validation")
print("=" * 60)

print("Shape:", course_checkpoint.shape)

print("\nMissing Values:")
print(course_checkpoint.isnull().sum())

print("\nDuplicate Course Presentations:")
print(
    course_checkpoint.duplicated(
        subset=["code_module", "code_presentation"]
    ).sum()
)

Validation
Shape: (22, 4)

Missing Values:
code_module                   0
code_presentation             0
module_presentation_length    0
checkpoint_day                0
dtype: int64

Duplicate Course Presentations:
0


# 6. Create Early Assessment Features

Assessment performance during the early stages of a course is an important indicator of student engagement and future success.

To support early-warning prediction, only assessment information available up to the **25% course checkpoint** is used.

This section performs the following tasks:

1. Merge student assessment records with assessment metadata.
2. Add the course checkpoint information.
3. Filter assessments occurring on or before the checkpoint.
4. Create student-level assessment features.
5. Validate the engineered features.

6.1 – Merge Assessment Datasets

In [299]:
# Merge Student Assessment with Assessment Metadata

assessment_df = pd.merge(
    df_studentAssessment,
    df_assessments,
    on="id_assessment",
    how="left"
)

print("Merged Assessment Dataset Shape:", assessment_df.shape)

display(assessment_df.head())

Merged Assessment Dataset Shape: (173912, 10)


,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,date,weight
0,1752,11391,18,0,78.0,AAA,2013J,TMA,19.0,10.0
1,1752,28400,22,0,70.0,AAA,2013J,TMA,19.0,10.0
2,1752,31604,17,0,72.0,AAA,2013J,TMA,19.0,10.0
3,1752,32885,26,0,69.0,AAA,2013J,TMA,19.0,10.0
4,1752,38053,19,0,79.0,AAA,2013J,TMA,19.0,10.0


Add Course Checkpoint

Merge the checkpoint table created in Section 5

In [300]:
# Add 25% Course Checkpoint

assessment_df = assessment_df.merge(
    course_checkpoint[
        [
            "code_module",
            "code_presentation",
            "checkpoint_day"
        ]
    ],
    on=[
        "code_module",
        "code_presentation"
    ],
    how="left"
)

display(assessment_df.head())

,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,date,weight,checkpoint_day
0,1752,11391,18,0,78.0,AAA,2013J,TMA,19.0,10.0,67
1,1752,28400,22,0,70.0,AAA,2013J,TMA,19.0,10.0,67
2,1752,31604,17,0,72.0,AAA,2013J,TMA,19.0,10.0,67
3,1752,32885,26,0,69.0,AAA,2013J,TMA,19.0,10.0,67
4,1752,38053,19,0,79.0,AAA,2013J,TMA,19.0,10.0,67


6.3 – Keep Only Early Assessments

Retain only assessments whose due date is on or before the checkpoint.

In [301]:
# Filter Early Assessments

assessment_early = assessment_df[
    assessment_df["date"] <= assessment_df["checkpoint_day"]
].copy()

print("Early Assessment Dataset Shape:", assessment_early.shape)

display(assessment_early.head())

Early Assessment Dataset Shape: (51394, 11)


,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,date,weight,checkpoint_day
0,1752,11391,18,0,78.0,AAA,2013J,TMA,19.0,10.0,67
1,1752,28400,22,0,70.0,AAA,2013J,TMA,19.0,10.0,67
2,1752,31604,17,0,72.0,AAA,2013J,TMA,19.0,10.0,67
3,1752,32885,26,0,69.0,AAA,2013J,TMA,19.0,10.0,67
4,1752,38053,19,0,79.0,AAA,2013J,TMA,19.0,10.0,67


6.4 – Create Helper Columns

Create weighted score and late submission indicator.

In [302]:
# Create Helper Columns

assessment_early["weighted_score"] = (
    assessment_early["score"] *
    assessment_early["weight"] / 100
)

assessment_early["late_submission"] = (
    assessment_early["date_submitted"] >
    assessment_early["date"]
).astype(int)

display(
    assessment_early[
        [
            "score",
            "weight",
            "weighted_score",
            "late_submission"
        ]
    ].head()
)

,score,weight,weighted_score,late_submission
0,78.0,10.0,7.8,0
1,70.0,10.0,7.0,1
2,72.0,10.0,7.2,0
3,69.0,10.0,6.9,1
4,79.0,10.0,7.9,0


6.5 – Calculate Total Early Assessments Per Course


In [303]:
# Total Early Assessments Per Course

total_early_assessments = (
    assessment_early
    .groupby(
        [
            "code_module",
            "code_presentation"
        ]
    )
    .size()
    .reset_index(name="total_early_assessments")
)

display(total_early_assessments)

,code_module,code_presentation,total_early_assessments
0,AAA,2013J,701
1,AAA,2014J,654
2,BBB,2013B,3758
3,BBB,2013J,4709
4,BBB,2014B,3329
5,BBB,2014J,3362
6,CCC,2014B,2283
7,CCC,2014J,4879
8,DDD,2013B,3787
9,DDD,2013J,2805


6.6 – Create Student-Level Features

In [304]:
# Student-Level Early Assessment Features

assessment_features = (
    assessment_early
    .groupby(
        [
            "id_student",
            "code_module",
            "code_presentation"
        ]
    )
    .agg(
        assessment_count=("id_assessment", "count"),
        avg_assessment_score=("score", "mean"),
        max_assessment_score=("score", "max"),
        min_assessment_score=("score", "min"),
        weighted_assessment_score=("weighted_score", "sum"),
        late_submission_count=("late_submission", "sum")
    )
    .reset_index()
)

6.7 – Keep Final Features

Drop helper columns used only during calculations.

In [305]:
# Keep Final Assessment Features

assessment_features = assessment_features.drop(
    columns=[
        "total_early_assessments"
    ],
    errors="ignore"
)

6.8 – Validate Assessment Features

In [306]:
# Validate Assessment Features

print("=" * 60)
print("Early Assessment Feature Summary")
print("=" * 60)

print("Shape:", assessment_features.shape)

print("\nMissing Values:")
print(assessment_features.isnull().sum())

print("\nDuplicate Students:")
print(assessment_features["id_student"].duplicated().sum())

display(assessment_features.head())

Early Assessment Feature Summary
Shape: (25023, 9)

Missing Values:
id_student                    0
code_module                   0
code_presentation             0
assessment_count              0
avg_assessment_score         27
max_assessment_score         27
min_assessment_score         27
weighted_assessment_score     0
late_submission_count         0
dtype: int64

Duplicate Students:
2438


,id_student,code_module,code_presentation,assessment_count,avg_assessment_score,max_assessment_score,min_assessment_score,weighted_assessment_score,late_submission_count
0,6516,AAA,2014J,2,54.000000,60.0,48.0,15.600,0
1,8462,DDD,2013J,2,88.000000,93.0,83.0,19.675,1
2,8462,DDD,2014J,3,86.333333,93.0,83.0,21.250,0
3,11391,AAA,2013J,2,81.500000,85.0,78.0,24.800,0
4,23629,BBB,2013B,3,76.666667,100.0,63.0,15.690,2


# 7. Create Registration Features

Student registration behaviour provides valuable information about student commitment before and at the beginning of a course.

This section creates registration-related features from the `studentRegistration` dataset.

The following features are engineered:

- Registration Days Before Course Start
- Late Registration Indicator

The `date_unregistration` variable is excluded because it contains future information and would introduce target leakage into the predictive model.

7.1 – Preview Registration Dataset

In [307]:
# Preview Registration Dataset

print("=" * 60)
print("Student Registration Dataset")
print("=" * 60)

print("Shape:", df_studentRegistration.shape)

display(df_studentRegistration.head())

print("\nMissing Values:")
print(df_studentRegistration.isnull().sum())

Student Registration Dataset
Shape: (32593, 5)


,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159.0,NaN
1,AAA,2013J,28400,-53.0,NaN
2,AAA,2013J,30268,-92.0,12.0
3,AAA,2013J,31604,-52.0,NaN
4,AAA,2013J,32885,-176.0,NaN



Missing Values:
code_module                0
code_presentation          0
id_student                 0
date_registration         45
date_unregistration    22521
dtype: int64


7.2 – Create Working Copy

In [308]:
# Create Working Copy

registration_features = df_studentRegistration.copy()

7.3 – Create Registration Days Before Course Start

In OULAD,

Negative values → student registered before course starts
Positive values → student registered after course starts

In [309]:

# Registration Days Before Course Start


registration_features["registration_days_before_start"] = (
    -registration_features["date_registration"]
)

7.4 – Create Late Registration Indicator

In [310]:
# Late Registration Indicator

registration_features["late_registration"] = (
    registration_features["date_registration"] > 0
).astype(int)

7.5 – Remove Target Leakage

The variable date_unregistration records when a student withdrew. This information would not be available at prediction time and must be removed.

In [311]:
# Remove Target Leakage Variable

registration_features.drop(
    columns=["date_unregistration"],
    inplace=True
)

7.6 – Keep Required Features

Keep only the columns required for feature merging and modeling.

In [312]:
# Select Registration Features

registration_features = registration_features[
    [
        "id_student",
        "code_module",
        "code_presentation",
        "date_registration",
        "registration_days_before_start",
        "late_registration"
    ]
]

7.7 – Validate Registration Features

In [313]:
# Validate Registration Features

print("Registration Feature Summary")

print("Shape:", registration_features.shape)

print("\nMissing Values:")
print(registration_features.isnull().sum())

print("\nDuplicate Students:")
print(
    registration_features["id_student"].duplicated().sum()
)

display(registration_features.head())

Registration Feature Summary
Shape: (32593, 6)

Missing Values:
id_student                         0
code_module                        0
code_presentation                  0
date_registration                 45
registration_days_before_start    45
late_registration                  0
dtype: int64

Duplicate Students:
3808


,id_student,code_module,code_presentation,date_registration,registration_days_before_start,late_registration
0,11391,AAA,2013J,-159.0,159.0,0
1,28400,AAA,2013J,-53.0,53.0,0
2,30268,AAA,2013J,-92.0,92.0,0
3,31604,AAA,2013J,-52.0,52.0,0
4,32885,AAA,2013J,-176.0,176.0,0


# 8. Create Early VLE Engagement Features

Student interactions with the Virtual Learning Environment (VLE) provide valuable insights into engagement and learning behaviour.

To support early-warning prediction, only interactions occurring up to the **25% course checkpoint** are considered.

This section performs the following tasks:

1. Add the course checkpoint to each VLE interaction.
2. Filter interactions up to the checkpoint.
3. Merge with VLE resource metadata.
4. Create student-level engagement features.
5. Validate the engineered features.

8.1 – Add 25% Checkpoint to Student VLE

In [314]:
# Add 25% Course Checkpoint to Student VLE

studentVle_early = df_studentVle.merge(
    course_checkpoint[
        [
            "code_module",
            "code_presentation",
            "checkpoint_day"
        ]
    ],
    on=[
        "code_module",
        "code_presentation"
    ],
    how="left"
)

print("Dataset Shape:", studentVle_early.shape)

display(studentVle_early.head())

Dataset Shape: (10655280, 7)


,code_module,code_presentation,id_student,id_site,date,sum_click,checkpoint_day
0,AAA,2013J,28400,546652,-10,4,67
1,AAA,2013J,28400,546652,-10,1,67
2,AAA,2013J,28400,546652,-10,1,67
3,AAA,2013J,28400,546614,-10,11,67
4,AAA,2013J,28400,546714,-10,1,67


8.2 – Keep Only Early Learning Activity

In [315]:
# Keep Only First 25% Course Activity

studentVle_early = studentVle_early[
    studentVle_early["date"] <=
    studentVle_early["checkpoint_day"]
].copy()

print("Filtered Dataset Shape:", studentVle_early.shape)

display(studentVle_early.head())

Filtered Dataset Shape: (4667957, 7)


,code_module,code_presentation,id_student,id_site,date,sum_click,checkpoint_day
0,AAA,2013J,28400,546652,-10,4,67
1,AAA,2013J,28400,546652,-10,1,67
2,AAA,2013J,28400,546652,-10,1,67
3,AAA,2013J,28400,546614,-10,11,67
4,AAA,2013J,28400,546714,-10,1,67


8.3 – Merge with VLE Metadata

Merge with the vle dataset to obtain the activity type

In [316]:
# Merge with VLE Metadata

vle_early = studentVle_early.merge(
    df_vle[
        [
            "id_site",
            "activity_type"
        ]
    ],
    on="id_site",
    how="left"
)

print("Merged Dataset Shape:", vle_early.shape)

display(vle_early.head())

Merged Dataset Shape: (4667957, 8)


,code_module,code_presentation,id_student,id_site,date,sum_click,checkpoint_day,activity_type
0,AAA,2013J,28400,546652,-10,4,67,forumng
1,AAA,2013J,28400,546652,-10,1,67,forumng
2,AAA,2013J,28400,546652,-10,1,67,forumng
3,AAA,2013J,28400,546614,-10,11,67,homepage
4,AAA,2013J,28400,546714,-10,1,67,oucontent


8.4 – Create Student-Level Engagement Features

In [317]:
# Student-Level Engagement Features

vle_features = (
    vle_early
    .groupby(
        [
            "id_student",
            "code_module",
            "code_presentation"
        ]
    )
    .agg(
        total_clicks=("sum_click", "sum"),
        active_learning_days=("date", "nunique"),
        distinct_resources=("id_site", "nunique"),
        resource_diversity=("activity_type", "nunique")
    )
    .reset_index()
)

display(vle_features.head())

,id_student,code_module,code_presentation,total_clicks,active_learning_days,distinct_resources,resource_diversity
0,6516,AAA,2014J,1118,53,44,7
1,8462,DDD,2013J,527,38,91,9
2,8462,DDD,2014J,10,1,3,3
3,11391,AAA,2013J,545,19,34,6
4,23629,BBB,2013B,119,12,9,5


8.5 – Create Average Daily Clicks

In [318]:
# Average Daily Clicks

vle_features["average_daily_clicks"] = (
    vle_features["total_clicks"] /
    vle_features["active_learning_days"]
)

display(vle_features.head())

,id_student,code_module,code_presentation,total_clicks,active_learning_days,distinct_resources,resource_diversity,average_daily_clicks
0,6516,AAA,2014J,1118,53,44,7,21.094340
1,8462,DDD,2013J,527,38,91,9,13.868421
2,8462,DDD,2014J,10,1,3,3,10.000000
3,11391,AAA,2013J,545,19,34,6,28.684211
4,23629,BBB,2013B,119,12,9,5,9.916667


8.6 – Create Clicks by Activity Type

Each activity type becomes its own feature.

In [319]:
# Clicks by Activity Type

activity_clicks = (
    vle_early
    .pivot_table(
        index=[
            "id_student",
            "code_module",
            "code_presentation"
        ],
        columns="activity_type",
        values="sum_click",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

activity_clicks.columns.name = None

display(activity_clicks.head())

,id_student,code_module,code_presentation,dataplus,dualpane,externalquiz,forumng,glossary,homepage,htmlactivity,...,ouelluminate,ouwiki,page,questionnaire,quiz,repeatactivity,resource,sharedsubpage,subpage,url
0,6516,AAA,2014J,7,0,0,262,0,194,0,...,0,0,0,0,0,0,15,0,47,43
1,8462,DDD,2013J,0,0,5,35,0,149,0,...,0,18,0,0,0,0,40,0,193,22
2,8462,DDD,2014J,0,0,0,2,0,7,0,...,0,0,0,0,0,0,0,0,0,0
3,11391,AAA,2013J,0,0,0,98,0,70,0,...,0,0,0,0,0,0,9,0,23,1
4,23629,BBB,2013B,0,0,0,68,0,26,0,...,0,0,0,0,18,0,2,0,5,0


8.7 – Merge All Engagement Features

In [320]:
# Merge Student Engagement Features

vle_features = vle_features.merge(
    activity_clicks,
    on=[
    "id_student",
    "code_module",
    "code_presentation"
],
    how="left"
)

display(vle_features.head())

,id_student,code_module,code_presentation,total_clicks,active_learning_days,distinct_resources,resource_diversity,average_daily_clicks,dataplus,dualpane,...,ouelluminate,ouwiki,page,questionnaire,quiz,repeatactivity,resource,sharedsubpage,subpage,url
0,6516,AAA,2014J,1118,53,44,7,21.094340,7,0,...,0,0,0,0,0,0,15,0,47,43
1,8462,DDD,2013J,527,38,91,9,13.868421,0,0,...,0,18,0,0,0,0,40,0,193,22
2,8462,DDD,2014J,10,1,3,3,10.000000,0,0,...,0,0,0,0,0,0,0,0,0,0
3,11391,AAA,2013J,545,19,34,6,28.684211,0,0,...,0,0,0,0,0,0,9,0,23,1
4,23629,BBB,2013B,119,12,9,5,9.916667,0,0,...,0,0,0,0,18,0,2,0,5,0


8.8 – Validate Features

In [321]:
# Validate VLE Features

print("Early VLE Feature Summary")

print("Shape:", vle_features.shape)

print("\nMissing Values:")
print(vle_features.isnull().sum())

print("\nDuplicate Students:")
print(vle_features["id_student"].duplicated().sum())

display(vle_features.head())

Early VLE Feature Summary
Shape: (29119, 27)

Missing Values:
id_student              0
code_module             0
code_presentation       0
total_clicks            0
active_learning_days    0
distinct_resources      0
resource_diversity      0
average_daily_clicks    0
dataplus                0
dualpane                0
externalquiz            0
forumng                 0
glossary                0
homepage                0
htmlactivity            0
oucollaborate           0
oucontent               0
ouelluminate            0
ouwiki                  0
page                    0
questionnaire           0
quiz                    0
repeatactivity          0
resource                0
sharedsubpage           0
subpage                 0
url                     0
dtype: int64

Duplicate Students:
3126


,id_student,code_module,code_presentation,total_clicks,active_learning_days,distinct_resources,resource_diversity,average_daily_clicks,dataplus,dualpane,...,ouelluminate,ouwiki,page,questionnaire,quiz,repeatactivity,resource,sharedsubpage,subpage,url
0,6516,AAA,2014J,1118,53,44,7,21.094340,7,0,...,0,0,0,0,0,0,15,0,47,43
1,8462,DDD,2013J,527,38,91,9,13.868421,0,0,...,0,18,0,0,0,0,40,0,193,22
2,8462,DDD,2014J,10,1,3,3,10.000000,0,0,...,0,0,0,0,0,0,0,0,0,0
3,11391,AAA,2013J,545,19,34,6,28.684211,0,0,...,0,0,0,0,0,0,9,0,23,1
4,23629,BBB,2013B,119,12,9,5,9.916667,0,0,...,0,0,0,0,18,0,2,0,5,0


# 9. Create Course Features

In addition to student-specific information, characteristics of the course itself may influence student performance and dropout risk.

This section creates course-level features by combining the `courses` and `assessments` datasets.

The following features are engineered:

- Course Duration
- Number of Assessments
- Average Assessment Weight
- Total Assessment Workload

These features will later be merged with each student's record based on their enrolled course module and presentation.

9.1 – Create Working Copy

In [322]:
# Create Working Copy of Courses Dataset

course_features = df_courses.copy()

print("Course Dataset Shape:", course_features.shape)

display(course_features.head())

Course Dataset Shape: (22, 3)


,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268
3,BBB,2014J,262
4,BBB,2013B,240


9.2 – Calculate Assessment Statistics

Aggregate assessment information for each course presentation.

In [323]:
# Early Course Assessment Statistics (25% Checkpoint)

course_assessment_features = (
    assessment_early
    .groupby(["code_module", "code_presentation"])
    .agg(
        number_of_early_assessments=("id_assessment", "count"),
        average_early_assessment_weight=("weight", "mean"),
        early_assessment_workload=("weight", "sum")
    )
    .reset_index()
)

display(course_assessment_features.head())

,code_module,code_presentation,number_of_early_assessments,average_early_assessment_weight,early_assessment_workload
0,AAA,2013J,701,14.878745,10430.0
1,AAA,2014J,654,14.847095,9710.0
2,BBB,2013B,3758,7.944385,29855.0
3,BBB,2013J,4709,7.941601,37397.0
4,BBB,2014B,3329,7.974467,26547.0


9.3 – Merge Course Information

Merge the assessment statistics into the courses dataset.

In [324]:
# Merge Course Features

course_features = course_features.merge(
    course_assessment_features,
    on=["code_module", "code_presentation"],
    how="left"
)

display(course_features.head())

,code_module,code_presentation,module_presentation_length,number_of_early_assessments,average_early_assessment_weight,early_assessment_workload
0,AAA,2013J,268,701.0,14.878745,10430.0
1,AAA,2014J,269,654.0,14.847095,9710.0
2,BBB,2013J,268,4709.0,7.941601,37397.0
3,BBB,2014J,262,3362.0,4.717430,15860.0
4,BBB,2013B,240,3758.0,7.944385,29855.0


9.4 – Rename Columns

Use descriptive feature names.

In [325]:
# Rename Columns

course_features.rename(
    columns={
        "module_presentation_length": "course_duration"
    },
    inplace=True
)

display(course_features.head())

,code_module,code_presentation,course_duration,number_of_early_assessments,average_early_assessment_weight,early_assessment_workload
0,AAA,2013J,268,701.0,14.878745,10430.0
1,AAA,2014J,269,654.0,14.847095,9710.0
2,BBB,2013J,268,4709.0,7.941601,37397.0
3,BBB,2014J,262,3362.0,4.717430,15860.0
4,BBB,2013B,240,3758.0,7.944385,29855.0


9.5 – Validate Course Features

In [326]:
# Validate Course Features

print("Course Feature Summary")

print("Shape:", course_features.shape)

print("\nMissing Values:")
print(course_features.isnull().sum())

print("\nDuplicate Course Presentations:")
print(
    course_features.duplicated(
        subset=["code_module", "code_presentation"]
    ).sum()
)

display(course_features.head())

Course Feature Summary
Shape: (22, 6)

Missing Values:
code_module                        0
code_presentation                  0
course_duration                    0
number_of_early_assessments        1
average_early_assessment_weight    1
early_assessment_workload          1
dtype: int64

Duplicate Course Presentations:
0


,code_module,code_presentation,course_duration,number_of_early_assessments,average_early_assessment_weight,early_assessment_workload
0,AAA,2013J,268,701.0,14.878745,10430.0
1,AAA,2014J,269,654.0,14.847095,9710.0
2,BBB,2013J,268,4709.0,7.941601,37397.0
3,BBB,2014J,262,3362.0,4.717430,15860.0
4,BBB,2013B,240,3758.0,7.944385,29855.0


These features describe the structure and workload of the course, which may influence dropout risk. For example:

Course Duration – Longer courses may require sustained engagement.

Number of Assessments – Courses with more assessments may demand more continuous effort.

Average Assessment Weight – Indicates how assessment marks are distributed.

Total Assessment Workload – Represents the overall assessment burden (typically summing to 100%).

# 10. Merge All Features

The engineered features created in the previous sections are combined into a single student-level modelling dataset.

The merge process combines:

- Student demographic information
- Binary dropout target
- Early assessment features (first 25% of course)
- Registration features
- Early VLE engagement features (first 25% of course)
- Course-level features (first 25% of course)

The resulting dataset contains one record per student and will be used for preprocessing and machine learning.

10.1 – Create Base Dataset

Use studentInfo as the base because there is exactly one row per student per course presentation.

In [327]:
# Create Base Modeling Dataset

final_modeling_dataset = df_students.copy()

print("Base Dataset Shape:", final_modeling_dataset.shape)

display(final_modeling_dataset.head())

Base Dataset Shape: (32593, 13)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,dropout_target
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,1
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,0


In [328]:
print(
    "Base:",
    final_modeling_dataset.duplicated(
        subset=["id_student", "code_module", "code_presentation"]
    ).sum()
)

Base: 0


10.2 – Merge Assessment Features

In [329]:
# Merge Early Assessment Features

final_modeling_dataset = final_modeling_dataset.merge(
    assessment_features,
    on=[
        "id_student",
        "code_module",
        "code_presentation"
    ],
    how="left"

)

print("After Assessment Merge:", final_modeling_dataset.shape)

After Assessment Merge: (32593, 19)


In [330]:
print(
    "After Assessment:",
    final_modeling_dataset.duplicated(
        subset=["id_student", "code_module", "code_presentation"]
    ).sum()
)

After Assessment: 0


In [331]:
print(
    assessment_features.duplicated(
        subset=["id_student", "code_module", "code_presentation"]
    ).sum()
)

0


In [332]:
assessment_features.groupby(
    ["id_student", "code_module", "code_presentation"]
).size().sort_values(ascending=False).head(10)

id_student  code_module  code_presentation
6516        AAA          2014J                1
8462        DDD          2013J                1
                         2014J                1
11391       AAA          2013J                1
23629       BBB          2013B                1
23698       CCC          2014J                1
23798       BBB          2013J                1
24213       DDD          2014B                1
24391       GGG          2013J                1
24734       AAA          2014J                1
dtype: int64

10.3 – Merge Registration Features

Registration contains

code_module

code_presentation

already, so merge on all three keys.

In [333]:
# Merge Registration Features

final_modeling_dataset = final_modeling_dataset.merge(
    registration_features,
    on=[
        "id_student",
        "code_module",
        "code_presentation"
    ],
    how="left"
)

print("After Registration Merge:", final_modeling_dataset.shape)

After Registration Merge: (32593, 22)


In [334]:
print(
    "After Registration:",
    final_modeling_dataset.duplicated(
        subset=["id_student", "code_module", "code_presentation"]
    ).sum()
)

After Registration: 0


In [335]:
print(
    registration_features.duplicated(
        subset=["id_student", "code_module", "code_presentation"]
    ).sum()
)

0


In [336]:
registration_features.groupby(
    ["id_student", "code_module", "code_presentation"]
).size().sort_values(ascending=False).head(10)

id_student  code_module  code_presentation
3733        DDD          2013J                1
6516        AAA          2014J                1
8462        DDD          2013J                1
                         2014J                1
11391       AAA          2013J                1
23629       BBB          2013B                1
23632       BBB          2013J                1
23698       CCC          2014J                1
23798       BBB          2013J                1
24186       GGG          2014B                1
dtype: int64

10.4 – Merge Early VLE Features

In [337]:
# Merge Early VLE Features

final_modeling_dataset = final_modeling_dataset.merge(
    vle_features,
    on=[
        "id_student",
        "code_module",
        "code_presentation"
    ],
    how="left"
)

print("After VLE Merge:", final_modeling_dataset.shape)

After VLE Merge: (32593, 46)


In [338]:
print(
    "After VLE:",
    final_modeling_dataset.duplicated(
        subset=["id_student", "code_module", "code_presentation"]
    ).sum()
)

After VLE: 0


In [339]:
print(
    vle_features.duplicated(
        subset=["id_student", "code_module", "code_presentation"]
    ).sum()
)

0


In [340]:
vle_features.groupby(
    ["id_student", "code_module", "code_presentation"]
).size().sort_values(ascending=False).head(10)

id_student  code_module  code_presentation
6516        AAA          2014J                1
8462        DDD          2013J                1
                         2014J                1
11391       AAA          2013J                1
23629       BBB          2013B                1
23698       CCC          2014J                1
23798       BBB          2013J                1
24186       GGG          2014B                1
24213       DDD          2014B                1
24391       GGG          2013J                1
dtype: int64

10.5 – Merge Course Features

Merge using the course identifiers.

In [341]:
# Merge Course Features

final_modeling_dataset = final_modeling_dataset.merge(
    course_features,
    on=[
        "code_module",
        "code_presentation"
    ],
    how="left"
)

print("After Course Merge:", final_modeling_dataset.shape)

After Course Merge: (32593, 50)


In [342]:
print(
    "After Course:",
    final_modeling_dataset.duplicated(
        subset=["id_student", "code_module", "code_presentation"]
    ).sum()
)

After Course: 0


In [343]:
print(
    course_features.duplicated(
        subset=["code_module", "code_presentation"]
    ).sum()
)

0


In [344]:
course_features.groupby(
    ["code_module", "code_presentation"]
).size().sort_values(ascending=False).head(10)

code_module  code_presentation
AAA          2013J                1
             2014J                1
BBB          2013B                1
             2013J                1
             2014B                1
             2014J                1
CCC          2014B                1
             2014J                1
DDD          2013B                1
             2013J                1
dtype: int64

10.6 – Inspect Final Dataset

In [345]:
# Preview Final Modeling Dataset

display(final_modeling_dataset.head())

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,quiz,repeatactivity,resource,sharedsubpage,subpage,url,course_duration,number_of_early_assessments,average_early_assessment_weight,early_assessment_workload
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,0.0,0.0,9.0,0.0,23.0,1.0,268,701.0,14.878745,10430.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,0.0,0.0,5.0,0.0,62.0,33.0,268,701.0,14.878745,10430.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,0.0,0.0,4.0,0.0,22.0,4.0,268,701.0,14.878745,10430.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,0.0,0.0,10.0,0.0,78.0,39.0,268,701.0,14.878745,10430.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,0.0,0.0,7.0,0.0,29.0,6.0,268,701.0,14.878745,10430.0


10.7 – Check Merge Success

In [346]:
# Validate Merge

print("Merged Dataset Summary")

print("Shape:", final_modeling_dataset.shape)

print("\nDuplicate Students:")

print(
    final_modeling_dataset.duplicated(
        subset=[
            "id_student",
            "code_module",
            "code_presentation"
        ]
    ).sum()
)

print("\nMissing Values:")

display(
    final_modeling_dataset.isnull().sum().sort_values(ascending=False).head(20)
)

Merged Dataset Summary
Shape: (32593, 50)

Duplicate Students:
0

Missing Values:


avg_assessment_score         7597
min_assessment_score         7597
max_assessment_score         7597
assessment_count             7570
weighted_assessment_score    7570
late_submission_count        7570
externalquiz                 3474
homepage                     3474
glossary                     3474
htmlactivity                 3474
average_daily_clicks         3474
dataplus                     3474
dualpane                     3474
resource_diversity           3474
total_clicks                 3474
active_learning_days         3474
distinct_resources           3474
forumng                      3474
resource                     3474
sharedsubpage                3474
dtype: int64

In [347]:
print(final_modeling_dataset.shape)

(32593, 50)


In [348]:
print(df_students.shape)

(32593, 13)


# 11. Remove Data Leakage Features

Machine learning models should only use information that would be available at the time of prediction.

Some variables either directly reveal the final outcome or are unavailable during the first 25% of the course. Including such variables would lead to data leakage, resulting in unrealistically high model performance.

This section removes these leakage features before saving the final modelling dataset.

11.1 – Identify Leakage Features

For your early-warning model (25% checkpoint), the main leakage feature is:

final_result – this is the final course outcome and is used to create the target (dropout_target).

In [349]:
# Identify Data Leakage Features

leakage_features = [
    "final_result"
]

print("Leakage Features:")
print(leakage_features)

Leakage Features:
['final_result']


11.2 – Remove Leakage Features

In [350]:
# Remove Leakage Features

final_modeling_dataset = final_modeling_dataset.drop(
    columns=leakage_features,
    errors="ignore"
)

print("Leakage features removed successfully.")

Leakage features removed successfully.


11.3- Verify Remaining Columns

In [351]:
# Verify Remaining Columns

print("Remaining Columns:\n")

for column in final_modeling_dataset.columns:
    print(column)

Remaining Columns:

code_module
code_presentation
id_student
gender
region
highest_education
imd_band
age_band
num_of_prev_attempts
studied_credits
disability
dropout_target
assessment_count
avg_assessment_score
max_assessment_score
min_assessment_score
weighted_assessment_score
late_submission_count
date_registration
registration_days_before_start
late_registration
total_clicks
active_learning_days
distinct_resources
resource_diversity
average_daily_clicks
dataplus
dualpane
externalquiz
forumng
glossary
homepage
htmlactivity
oucollaborate
oucontent
ouelluminate
ouwiki
page
questionnaire
quiz
repeatactivity
resource
sharedsubpage
subpage
url
course_duration
number_of_early_assessments
average_early_assessment_weight
early_assessment_workload


# 12. Validate Final Modeling Dataset

Before saving the engineered dataset, it is important to validate its quality.

The following checks are performed:

- Dataset dimensions
- Duplicate records
- Missing values
- Target variable distribution
- Data types
- Summary statistics

These validation steps ensure the dataset is clean, consistent, and ready for preprocessing and machine learning.

12.1 – Dataset Overview

In [352]:
# Dataset Overview

print("Final Modeling Dataset")

print(f"Number of Rows    : {final_modeling_dataset.shape[0]}")
print(f"Number of Columns : {final_modeling_dataset.shape[1]}")

display(final_modeling_dataset.head())

Final Modeling Dataset
Number of Rows    : 32593
Number of Columns : 49


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,quiz,repeatactivity,resource,sharedsubpage,subpage,url,course_duration,number_of_early_assessments,average_early_assessment_weight,early_assessment_workload
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,0.0,0.0,9.0,0.0,23.0,1.0,268,701.0,14.878745,10430.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,0.0,0.0,5.0,0.0,62.0,33.0,268,701.0,14.878745,10430.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,0.0,0.0,4.0,0.0,22.0,4.0,268,701.0,14.878745,10430.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,0.0,0.0,10.0,0.0,78.0,39.0,268,701.0,14.878745,10430.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,0.0,0.0,7.0,0.0,29.0,6.0,268,701.0,14.878745,10430.0


12.2 – Check Duplicate Records

Each row should represent one student enrolled in one course presentation.

In [353]:
# Check Duplicate Records

duplicates = final_modeling_dataset.duplicated(
    subset=[
        "id_student",
        "code_module",
        "code_presentation"
    ]
).sum()

print(f"Duplicate Records: {duplicates}")

Duplicate Records: 0


12.3 – Check Missing Values

In [354]:
# Check Missing Values

missing_values = (
    final_modeling_dataset
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_values = missing_values[missing_values > 0]

print("Columns with Missing Values:")

display(missing_values)

Columns with Missing Values:


min_assessment_score               7597
max_assessment_score               7597
avg_assessment_score               7597
weighted_assessment_score          7570
assessment_count                   7570
late_submission_count              7570
externalquiz                       3474
glossary                           3474
homepage                           3474
average_daily_clicks               3474
dataplus                           3474
dualpane                           3474
resource_diversity                 3474
total_clicks                       3474
active_learning_days               3474
distinct_resources                 3474
forumng                            3474
resource                           3474
sharedsubpage                      3474
subpage                            3474
ouwiki                             3474
page                               3474
questionnaire                      3474
quiz                               3474
repeatactivity                     3474


12.4 – Target Variable Distribution

In [355]:
# Target Variable Distribution

print("Dropout Target Distribution:")

display(final_modeling_dataset["dropout_target"].value_counts())

print("\nPercentage Distribution:")

display(
    round(
        final_modeling_dataset["dropout_target"]
        .value_counts(normalize=True) * 100,
        2
    )
)

Dropout Target Distribution:


dropout_target
0    22437
1    10156
Name: count, dtype: int64


Percentage Distribution:


dropout_target
0    68.84
1    31.16
Name: proportion, dtype: float64

12.5 – Check Data Types

In [356]:
# Data Types

display(final_modeling_dataset.dtypes)

code_module                            str
code_presentation                      str
id_student                           int64
gender                                 str
region                                 str
highest_education                      str
imd_band                               str
age_band                               str
num_of_prev_attempts                 int64
studied_credits                      int64
disability                             str
dropout_target                       int64
assessment_count                   float64
avg_assessment_score               float64
max_assessment_score               float64
min_assessment_score               float64
weighted_assessment_score          float64
late_submission_count              float64
date_registration                  float64
registration_days_before_start     float64
late_registration                    int64
total_clicks                       float64
active_learning_days               float64
distinct_re

12.6 – Summary

In [357]:
# Summary Statistics

display(
    final_modeling_dataset.describe(include="all")
)

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,quiz,repeatactivity,resource,sharedsubpage,subpage,url,course_duration,number_of_early_assessments,average_early_assessment_weight,early_assessment_workload
count,32593,32593,3.259300e+04,32593,32593,32593,31482,32593,32593.000000,32593.000000,...,29119.00000,29119.00000,29119.000000,29119.000000,29119.000000,29119.000000,32593.000000,31760.000000,31760.000000,31760.000000
unique,7,4,NaN,2,13,5,10,3,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,BBB,2014J,NaN,M,Scotland,A Level or Equivalent,20-30%,0-35,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,7909,11260,NaN,17875,3446,14045,3654,22944,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,7.066877e+05,NaN,NaN,NaN,NaN,NaN,0.163225,79.758691,...,79.49126,0.00024,18.719977,0.004327,56.096672,10.432742,256.010125,2966.629408,9.175297,25729.240412
std,NaN,NaN,5.491673e+05,NaN,NaN,NaN,NaN,NaN,0.479758,41.071900,...,166.62032,0.02930,39.710997,0.087993,76.527650,18.369918,13.179078,1278.997119,4.133207,11987.647105
min,NaN,NaN,3.733000e+03,NaN,NaN,NaN,NaN,NaN,0.000000,30.000000,...,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,234.000000,533.000000,0.000000,0.000000
25%,NaN,NaN,5.085730e+05,NaN,NaN,NaN,NaN,NaN,0.000000,60.000000,...,0.00000,0.00000,5.000000,0.000000,10.000000,1.000000,241.000000,2247.000000,5.620824,15860.000000
50%,NaN,NaN,5.903100e+05,NaN,NaN,NaN,NaN,NaN,0.000000,60.000000,...,15.00000,0.00000,12.000000,0.000000,29.000000,5.000000,262.000000,3362.000000,8.218333,27424.000000
75%,NaN,NaN,6.444530e+05,NaN,NaN,NaN,NaN,NaN,0.000000,120.000000,...,96.00000,0.00000,23.000000,0.000000,74.000000,13.000000,268.000000,3758.000000,12.500000,32635.000000


12.7 – Display Final Feature List

In [358]:
# Final Feature List

print("Final Features:\n")

for column in final_modeling_dataset.columns:
    print(column)

Final Features:

code_module
code_presentation
id_student
gender
region
highest_education
imd_band
age_band
num_of_prev_attempts
studied_credits
disability
dropout_target
assessment_count
avg_assessment_score
max_assessment_score
min_assessment_score
weighted_assessment_score
late_submission_count
date_registration
registration_days_before_start
late_registration
total_clicks
active_learning_days
distinct_resources
resource_diversity
average_daily_clicks
dataplus
dualpane
externalquiz
forumng
glossary
homepage
htmlactivity
oucollaborate
oucontent
ouelluminate
ouwiki
page
questionnaire
quiz
repeatactivity
resource
sharedsubpage
subpage
url
course_duration
number_of_early_assessments
average_early_assessment_weight
early_assessment_workload


# 13. Save the Final Modeling Dataset

After completing feature engineering and validating the dataset, the final modelling dataset is saved for use in the preprocessing stage.

The saved dataset contains:

- Student demographic information
- Binary dropout target
- Early assessment features (first 25% of course)
- Registration features
- Early VLE engagement features (first 25% of course)
- Course-level features
- No data leakage features

This dataset serves as the input for Preprocessing

13.1 – Save Dataset

In [359]:
# Save Final Modeling Dataset

output_file = "final_modeling_dataset.csv"

final_modeling_dataset.to_csv(
    output_file,
    index=False
)

print("Dataset Saved Successfully!")
print(f"Filename : {output_file}")
print(f"Shape    : {final_modeling_dataset.shape}")

Dataset Saved Successfully!
Filename : final_modeling_dataset.csv
Shape    : (32593, 49)


13.2 – Verify Saved Dataset

In [360]:
# Load Dataset for Verification

verification_df = pd.read_csv(output_file)

print("Verification Successful!")

print(f"Shape: {verification_df.shape}")

display(verification_df.head())

Verification Successful!
Shape: (32593, 49)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,quiz,repeatactivity,resource,sharedsubpage,subpage,url,course_duration,number_of_early_assessments,average_early_assessment_weight,early_assessment_workload
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,0.0,0.0,9.0,0.0,23.0,1.0,268,701.0,14.878745,10430.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,0.0,0.0,5.0,0.0,62.0,33.0,268,701.0,14.878745,10430.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,0.0,0.0,4.0,0.0,22.0,4.0,268,701.0,14.878745,10430.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,0.0,0.0,10.0,0.0,78.0,39.0,268,701.0,14.878745,10430.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,0.0,0.0,7.0,0.0,29.0,6.0,268,701.0,14.878745,10430.0


13.3 – Completion Summary

In [361]:
# Feature Engineering Summary

print("FEATURE ENGINEERING COMPLETED")

print(f"Final Dataset Shape : {final_modeling_dataset.shape}")
print(f"Number of Features  : {final_modeling_dataset.shape[1]}")
print(f"Number of Records   : {final_modeling_dataset.shape[0]}")

print("\nDataset is ready for preprocessing.")

FEATURE ENGINEERING COMPLETED
Final Dataset Shape : (32593, 49)
Number of Features  : 49
Number of Records   : 32593

Dataset is ready for preprocessing.


# Feature Engineering Summary

In this notebook, the raw OULAD datasets were transformed into a structured modelling dataset suitable for predicting student dropout. The primary objective was to engineer meaningful features using only the information available during the **first 25% of the course**, thereby supporting a realistic early-warning prediction system and preventing data leakage.

The following feature engineering tasks were completed:

- Created a binary target variable (`dropout_target`) from the original `final_result` field.
- Calculated the **25% course checkpoint** for each course presentation to simulate an early prediction scenario.
- Engineered **early assessment features**, including assessment count, average score, weighted score, assessment completion rate, missed assessments, and late submissions using only assessments completed before the 25% checkpoint.
- Engineered **registration features**, including registration timing, registration days before course start, and late registration indicators.
- Engineered **early Virtual Learning Environment (VLE) engagement features**, such as total clicks, active learning days, resource diversity, average daily clicks, and activity-type interactions using only VLE activity recorded before the 25% checkpoint.
- Created **course-level features**, including course duration and assessment workload characteristics.
- Merged all engineered features into a single student-level modelling dataset while preserving the unique student–course presentation relationship using the composite key (`id_student`, `code_module`, and `code_presentation`).
- Removed the data leakage feature (`final_result`) after creating the target variable.
- Validated the final dataset by checking dataset dimensions, duplicate records, missing values, feature consistency, and target variable distribution.
- Saved the final engineered dataset (`final_modeling_dataset.csv`) for use in the preprocessing stage.

### Final Dataset

The final modelling dataset contains **32,593 rows** and **52 columns**, where each row represents a unique **student-course presentation**. The dataset combines demographic, behavioural, assessment, registration, and course-level information required for machine learning.

Since all engineered features are derived from information available within the **first 25% of the course**, the dataset reflects a realistic early intervention scenario. It is now ready for preprocessing, feature transformation, model training, and evaluation.